# 05 - Model 3: DistilBERT (fine-tuned) - Google Colab

**Self-contained.** This notebook imports nothing from `src/`, so it runs in Colab
on its own. It only needs the three CSVs from `data/processed/`.

### Before you run
1. **Runtime -> Change runtime type -> T4 GPU.** On CPU this takes hours.
2. Get `train.csv`, `val.csv` and `test.csv` into the session (cell 2 offers upload
   or Google Drive).
3. Run the install cell, then **restart the runtime** when told to.

### What it does
`distilbert-base-uncased`, `max_length=256`, 3 epochs, class-weighted loss for the
~5% positive rate. Scores the **same test split** as the local notebooks and writes
`predictions_distilbert_test.csv` plus a metrics row to paste into
`reports/experiments.csv`.

Expect roughly 10-20 minutes on a T4.

## 1. Install

Colab already ships a working torch + numpy + CUDA combination. Installing only
what is missing avoids the `numpy.dtype size changed` ABI error that comes from
pinning numpy or torch on top of Colab's build. **Do not** `pip install -r
requirements.txt` here.

In [2]:
!pip install -q "transformers==4.46.3" "accelerate==1.1.1"

print("\n" + "=" * 62)
print("RESTART THE RUNTIME NOW: Runtime -> Restart session")
print("Then skip this cell and carry on from cell 2.")
print("=" * 62)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 98.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.

RESTART THE RUNTIME NOW: Runtime -> Restart session
Then skip this cell and carry on from cell 2.


## 2. Get the data in

Either upload the three CSVs, or mount Drive if you put them there. `full_text` is
already built by `src/preprocessing.py`, so nothing is recomputed and the text is
byte-identical to what the local models saw.

In [1]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/fjpd_data")   # SAME splits as every other model
print("found:", sorted(p.name for p in DATA_DIR.glob("*.csv")))

Mounted at /content/drive
found: ['fake_job_postings.csv', 'predictions_baseline_tfidf_logreg_test.csv', 'predictions_lightgbm_tfidf_svd_meta_test.csv', 'predictions_minilm_frozen_lightgbm_test.csv', 'predictions_minilm_frozen_logreg_test.csv', 'test.csv', 'train.csv', 'val.csv']


## 3. Config - mirrors `src/config.py`

In [2]:
import random

import numpy as np
import pandas as pd
import torch

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
TEXT_COLUMN = "full_text"
TARGET = "fraudulent"
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1


def set_seed(seed=SEED):
    """Same seeding as src/config.set_seed, so the run reproduces."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

torch 2.11.0+cu128 | device: cuda
GPU: Tesla T4


In [3]:
def load_split(name):
    """Read a split and restore the empty strings a CSV round-trip turns into NaN."""
    frame = pd.read_csv(DATA_DIR / f"{name}.csv")
    frame[TEXT_COLUMN] = frame[TEXT_COLUMN].fillna("").astype(str)
    return frame


train = load_split("train")
val = load_split("val")
test = load_split("test")

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:>5}: {len(part):>6,} rows, {part[TARGET].mean():.2%} fraudulent")

train: 12,319 rows, 4.87% fraudulent
  val:  2,640 rows, 4.85% fraudulent
 test:  2,640 rows, 4.85% fraudulent


## 4. Tokenize

In [4]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class PostingsDataset(Dataset):
    """Tokenized postings. Truncation at 256 tokens keeps the title and the opening
    of the description, which is where most of the signal lives."""

    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding="max_length",
            max_length=MAX_LENGTH, return_tensors="pt",
        )
        self.labels = None if labels is None else torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, index):
        item = {key: value[index] for key, value in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[index]
        return item


train_ds = PostingsDataset(train[TEXT_COLUMN], train[TARGET])
val_ds = PostingsDataset(val[TEXT_COLUMN], val[TARGET])
test_ds = PostingsDataset(test[TEXT_COLUMN], test[TARGET])
print(f"tokenized: {len(train_ds):,} / {len(val_ds):,} / {len(test_ds):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenized: 12,319 / 2,640 / 2,640


## 5. Class-weighted trainer

With ~5% positives, plain cross-entropy lets the model collapse to "always real".
Weighting the loss inversely to class frequency is the transformer equivalent of
LightGBM's `scale_pos_weight`.

In [5]:
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

counts = np.bincount(train[TARGET], minlength=2)
class_weights = torch.tensor(len(train) / (2.0 * counts), dtype=torch.float)
print(f"class counts: real={counts[0]:,} fake={counts[1]:,}")
print(f"loss weights: real={class_weights[0]:.3f} fake={class_weights[1]:.3f}")


class WeightedTrainer(Trainer):
    """Trainer with a class-weighted cross-entropy loss.

    **kwargs absorbs num_items_in_batch, which newer transformers versions pass in
    and older ones do not.
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = F.cross_entropy(outputs.logits, labels,
                               weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

class counts: real=11,719 fake=600
loss weights: real=0.526 fake=10.266


In [6]:
set_seed()
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Evaluation is done manually after training, so no eval_strategy here - that
# argument was renamed across transformers versions and is easy to break on.
args = TrainingArguments(
    output_dir="/content/distilbert_out",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=100,
    save_strategy="no",
    seed=SEED,
    fp16=(DEVICE == "cuda"),
    report_to="none",
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds)
trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
100,0.650500
200,0.558700
300,0.651700
400,0.483300
500,0.503700
600,0.462300
700,0.333500
800,0.247800
900,0.235700
1000,0.297700


TrainOutput(global_step=2310, training_loss=0.28154861978638224, metrics={'train_runtime': 277.7952, 'train_samples_per_second': 133.037, 'train_steps_per_second': 8.315, 'total_flos': 2447798826064896.0, 'train_loss': 0.28154861978638224, 'epoch': 3.0})

## 6. Predict

In [7]:
from scipy.special import softmax


def predict_proba(dataset):
    """Probability of the positive (fraudulent) class."""
    logits = trainer.predict(dataset).predictions
    return softmax(logits, axis=1)[:, 1]


proba_val = predict_proba(val_ds)
proba_test = predict_proba(test_ds)
y_val = val[TARGET].to_numpy()
y_test = test[TARGET].to_numpy()
print("done")

done


## 7. Evaluate

Same protocol as the local notebooks: tune the threshold on validation, then report
test at that frozen value.

In [8]:
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score,
                             precision_recall_curve, precision_score, recall_score,
                             roc_auc_score)


def evaluate_predictions(y_true, y_proba, threshold=0.5):
    """Mirrors src/evaluation.evaluate_predictions so the numbers are comparable."""
    y_pred = (np.asarray(y_proba) >= threshold).astype(int)
    return {
        "average_precision": average_precision_score(y_true, y_proba),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "threshold": float(threshold),
    }


def tune_threshold(y_true, y_proba, beta=1.0):
    """Threshold maximising F-beta. Mirrors src/evaluation.tune_threshold."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    precision, recall = precision[:-1], recall[:-1]
    denominator = (beta ** 2 * precision) + recall
    scores = np.where(denominator > 0,
                      (1 + beta ** 2) * precision * recall / np.where(denominator == 0, 1, denominator),
                      0.0)
    best = int(np.argmax(scores))
    return float(thresholds[best]), float(scores[best])


best_threshold, best_f2 = tune_threshold(y_val, proba_val, beta=2.0)
print(f"best F2 threshold on val: {best_threshold:.3f} (F2={best_f2:.3f})\n")

val_metrics = evaluate_predictions(y_val, proba_val, best_threshold)
test_metrics = evaluate_predictions(y_test, proba_test, best_threshold)
print("val: ", {k: round(v, 4) for k, v in val_metrics.items()})
print("test:", {k: round(v, 4) for k, v in test_metrics.items()})

best F2 threshold on val: 0.021 (F2=0.894)

val:  {'average_precision': np.float64(0.9327), 'roc_auc': np.float64(0.9893), 'f1': 0.8755, 'precision': 0.8467, 'recall': 0.9062, 'accuracy': 0.9875, 'threshold': 0.0208}
test: {'average_precision': np.float64(0.8877), 'roc_auc': np.float64(0.9847), 'f1': 0.8154, 'precision': 0.803, 'recall': 0.8281, 'accuracy': 0.9818, 'threshold': 0.0208}


## 8. Export

Two artefacts go back to the repo:
* `predictions_distilbert_test.csv` -> drop into `models/` and notebook 04's
  comparison cell picks it up automatically.
* two CSV rows -> paste into `reports/experiments.csv`.

In [9]:
from datetime import datetime

predictions_path = "/content/predictions_distilbert_test.csv"
pd.DataFrame({"y_true": y_test, "y_proba": proba_test}).to_csv(predictions_path, index=False)

# Column order must match reports/experiments.csv exactly.
COLUMNS = ["timestamp", "model", "split", "average_precision", "roc_auc", "f1",
           "precision", "recall", "accuracy", "threshold", "notes"]
rows = []
for split, metrics, note in [("val", val_metrics, "DistilBERT 256tok/3ep, weighted loss, F2-tuned"),
                             ("test", test_metrics, "threshold frozen from val")]:
    row = {"timestamp": datetime.now().isoformat(timespec="seconds"),
           "model": "distilbert_finetuned", "split": split, "notes": note}
    row.update({k: metrics[k] for k in COLUMNS if k in metrics})
    rows.append(row)

log = pd.DataFrame(rows, columns=COLUMNS)
log.to_csv("/content/experiments_distilbert.csv", index=False)

print("Paste these two lines into reports/experiments.csv (no header):\n")
print(log.to_csv(index=False, header=False).strip())

Paste these two lines into reports/experiments.csv (no header):

2026-08-05T18:20:16,distilbert_finetuned,val,0.932727481639959,0.9892858031449043,0.8754716981132076,0.8467153284671532,0.90625,0.9875,0.020844316110014915,"DistilBERT 256tok/3ep, weighted loss, F2-tuned"
2026-08-05T18:20:16,distilbert_finetuned,test,0.8876914891459735,0.984732658240446,0.8153846153846154,0.803030303030303,0.828125,0.9818181818181818,0.020844316110014915,threshold frozen from val


## Back in the repo

1. Move `predictions_distilbert_test.csv` into `models/`.
2. Append the two rows to `reports/experiments.csv`.
3. Re-run the comparison cell in `04_sentence_transformers.ipynb` - DistilBERT now
   appears on the shared PR curve, scored on identical test rows.

## Conclusions

**Fine-tuning helps, but still does not beat bag-of-words here** - test PR-AUC 0.888, third of five: below the tabular LightGBM (0.921) and the TF-IDF baseline (0.909), above both frozen-MiniLM heads (0.836 / 0.545). Adapting the encoder's weights to the task buys a clear +0.05 over the best frozen embedding, so fine-tuning is doing real work - but not enough to overtake a plain TF-IDF logistic regression. The high ROC-AUC (0.985) next to the lower AP is the same lesson every model on this dataset repeats: under ~5% positives, ROC flatters; average precision is the metric that separates them.

**The likely ceiling is data, not architecture.** The encoder fine-tunes on ~600
positive examples against 66M parameters, and input is still truncated at 256 tokens - the same window as MiniLM - so the isolated variable versus the frozen model is the fine-tuning itself, not seeing more text. The sharp lexical triggers TF-IDF keeps explicit (`$5000`, `data entry`, `work from home`) are exactly what a small-positive, truncated transformer struggles to relearn from scratch.

**Practical takeaway.** On this dataset simple lexical signals are hard to beat even with a fine-tuned transformer - a genuinely useful negative result. The tuned F2 threshold lands at 0.021 (weighted loss pushes positive probabilities low), extending the 0.005-to-0.734 spread across models and reinforcing why 0.5 is never left as the default. For deployment, the tabular LightGBM remains the strongest single model; the transformer's value is confirmatory - it agrees the metadata-light, lexically sharp posts are the fraudulent ones.